### Instalação das dependências

O pacote evidently gera relatórios interativos de monitoramento de Data Drift e qualidade de dados. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install evidently pandas -q

# Código 3.2 — Detecção de Data Drift com Evidently AI

**Objetivo:** Identificar desvios na distribuição dos dados de entrada (*Data Drift*) comparando dados de referência (treino) com dados de produção recentes.

### Passo 1: Geração de Datasets Sintéticos (Referência vs Produção)

Simulamos uma mudança de distribuição na variável de temperatura para induzir o drift no relatório.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

Path("data").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

# 1. Dados de Referência (Distribuição normal original)
np.random.seed(42)
df_ref = pd.DataFrame({
    "temperatura": np.random.normal(loc=70, scale=5, size=500),
    "vibracao": np.random.normal(loc=10, scale=2, size=500),
    "pressao": np.random.normal(loc=100, scale=15, size=500),
})

# 2. Dados de Produção com Drift (Temperatura teve aumento médio e maior variabilidade)
np.random.seed(123)
df_prod = pd.DataFrame({
    "temperatura": np.random.normal(loc=82, scale=8, size=500),
    "vibracao": np.random.normal(loc=10.2, scale=2.1, size=500),
    "pressao": np.random.normal(loc=99, scale=14, size=500),
})

df_ref.to_csv("data/treino_referencia.csv", index=False)
df_prod.to_csv("data/producao_atual.csv", index=False)
print("Datasets de referência e produção gerados com sucesso!")

**Entendendo a simulação de Data Drift:**
- `df_ref`: Dataset de referência de treino com a temperatura variando em torno de 70°C (`loc=70`).
- `df_prod`: Dataset de produção simulando aquecimento no processo fabril, com temperatura média de 82°C (`loc=82`), configurando o Data Drift.

### Passo 2: Configuração e Execução do Relatório Evidently

Importamos a suíte de testes de Data Drift do `Evidently` para calcular métricas estatísticas de aderência (ex: Teste Kolmogorov-Smirnov).

In [ ]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Carregar datasets
ref = pd.read_csv("data/treino_referencia.csv")
prod = pd.read_csv("data/producao_atual.csv")

# Criar e executar relatório de Data Drift
report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=ref, current_data=prod)

# Salvar relatório HTML interativo
report_path = "reports/relatorio_drift.html"
report.save_html(report_path)
print(f"Relatório HTML de Data Drift salvo em: {report_path}")

**Entendendo os componentes do Evidently AI:**
- `Report(metrics=[DataDriftPreset()])`: Inicializa a suíte pré-configurada de métricas estatísticas de drift para cada coluna.
- `report.run(reference_data=..., current_data=...)`: Executa os testes de hipóteses estatísticas comparando a base de treino contra a base atual de produção.
- `save_html()`: Exporta o relatório completo com gráficos interativos em formato HTML.

---

## Conclusão da Aula — Detecção de Data Drift

O monitoramento contínuo com **Evidently AI** permite detectar quando a distribuição dos dados de entrada mudou em produção (*Data Drift*), sinalizando a necessidade de retreinamento antes que a acurácia do modelo caia significativamente.